<a href="https://colab.research.google.com/github/revo-off/CPVision-Fall-Detection/blob/main/notebook_mobile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![YOLO Illustration](https://cdn.prod.website-files.com/614c82ed388d53640613982e/65390c0119ee1e54a61cac91_evolution-yolo.webp)

---
# Installation & Configuration

In [ ]:
# Install core ML libraries
!pip install ultralytics opencv-python-headless Pillow matplotlib pandas numpy torch torchvision seaborn tqdm pyyaml nest_asyncio --quiet

# Install QAT support (requires NVIDIA GPU)
!pip install nvidia-modelopt --quiet 2>/dev/null || echo "⚠️ nvidia-modelopt not available — PTQ will be used as INT8 fallback."

# Fix asyncio for Jupyter/Colab (needed for NDJSON conversion)
import nest_asyncio
nest_asyncio.apply()

# Verify installations
import importlib

packages = [
    ('ultralytics', 'ultralytics'),
    ('cv2', 'opencv-python'),
    ('PIL', 'Pillow'),
    ('matplotlib', 'matplotlib'),
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('torch', 'torch'),
    ('torchvision', 'torchvision'),
    ('seaborn', 'seaborn'),
    ('yaml', 'pyyaml'),
]

print('\n📦 Checking installed packages:')
all_ok = True
for module_name, package_name in packages:
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, '__version__', 'OK')
        print(f'✅ {package_name:<25} v{version}')
    except ImportError:
        print(f'❌ {package_name:<25} NOT INSTALLED')
        all_ok = False

import torch
print(f'\n🖥️  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

print()
if all_ok:
    print('✅ All packages installed! Ready to train.')
else:
    print('⚠️ Some packages missing. Re-run this cell.')

---
# Dataset Preparation

| Property | Value |
|----------|-------|
| **Format** | NDJSON (Ultralytics native) |
| **Classes** | `0: fall`, `1: non-fall` |
| **Total Images** | ~17,900 |
| **Splits** | Train / Val / Test |

In [ ]:
import json
from pathlib import Path
from collections import Counter

# ── Dataset path ──────────────────────────────────────────────
# Adjust this path if running in Colab (upload the file or mount Drive)
NDJSON_PATH = 'data/merged-fall-dataset-yolo.ndjson'

# For Google Colab: uncomment the lines below to upload the file
from google.colab import files
uploaded = files.upload()  # Upload the .ndjson file
NDJSON_PATH = list(uploaded.keys())[0]

# ── Parse dataset ─────────────────────────────────────────────
with open(NDJSON_PATH, 'r') as f:
    lines = [json.loads(line) for line in f]

metadata = lines[0]
images = [l for l in lines[1:] if l.get('type') == 'image']

print(f"📊 Dataset: {metadata['name']}")
print(f"📝 {metadata['description']}")
print(f"🏷️  Classes: {metadata['class_names']}")
print(f"📦 Total images: {len(images)}")

# Split distribution
splits = Counter(img['split'] for img in images)
print(f"\n📂 Split distribution:")
for split, count in sorted(splits.items()):
    pct = count / len(images) * 100
    bar = '█' * int(pct / 2)
    print(f"   {split:>5}: {count:>5}  ({pct:.1f}%)  {bar}")

# Class distribution
class_counts = Counter()
for img in images:
    if 'annotations' in img:
        for box in img['annotations']['boxes']:
            class_counts[int(box[0])] += 1

print(f"\n🏷️  Annotation distribution:")
for cls_id, count in sorted(class_counts.items()):
    cls_name = metadata['class_names'][str(cls_id)]
    print(f"   {cls_name} (class {cls_id}): {count:,} bounding boxes")

total_annotations = sum(class_counts.values())
print(f"\n   Total annotations: {total_annotations:,}")
print(f"   Avg annotations/image: {total_annotations / len(images):.1f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import requests
from io import BytesIO
import random

# Pick 8 random images with annotations
annotated_images = [img for img in images if 'annotations' in img and img['annotations']['boxes']]
samples = random.sample(annotated_images, min(8, len(annotated_images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('📸 Sample Images from Fall Detection Dataset', fontsize=16, fontweight='bold')

class_colors = {0: '#FF4444', 1: '#44FF44'}  # fall=red, non-fall=green
class_labels = metadata['class_names']

for i, ax in enumerate(axes.flatten()):
    if i < len(samples):
        sample = samples[i]
        try:
            response = requests.get(sample['url'], timeout=10)
            img = Image.open(BytesIO(response.content))
            ax.imshow(img)

            w, h = sample['width'], sample['height']
            for box in sample['annotations']['boxes']:
                cls_id, cx, cy, bw, bh = int(box[0]), box[1], box[2], box[3], box[4]
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                rect = patches.Rectangle(
                    (x1, y1), bw * w, bh * h,
                    linewidth=2, edgecolor=class_colors[cls_id],
                    facecolor='none', linestyle='-'
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, class_labels[str(cls_id)],
                        color='white', fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2',
                                  facecolor=class_colors[cls_id], alpha=0.8))

            ax.set_title(f"{'🔴 FALL' if any(int(b[0]) == 0 for b in sample['annotations']['boxes']) else '🟢 OK'}",
                         fontsize=11)
        except Exception as e:
            ax.text(0.5, 0.5, f'Error loading\n{e}', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.show()
print("🔴 Red = Fall detected  |  🟢 Green = Non-fall (normal)")

In [ ]:
# ── Convert NDJSON → YOLO format ──────────────────────────────
# Jupyter/Colab runs its own asyncio event loop, so Ultralytics'
# automatic NDJSON conversion fails with "asyncio.run() cannot be
# called from a running event loop". We pre-convert here instead.

from ultralytics.data.converter import convert_ndjson_to_yolo
import asyncio

print("📦 Converting NDJSON → YOLO directory format...")
print("   (downloading images — this may take several minutes)\n")

DATA_YAML = await convert_ndjson_to_yolo(NDJSON_PATH)

print(f"\n✅ Conversion complete!")
print(f"📄 Dataset YAML: {DATA_YAML}")
print(f"\n   This YAML file will be used for all training & evaluation.")

---
# Model Training — FP32 (Standard Precision)

| Property | Value |
|----------|-------|
| **Model** | YOLO26n (Nano) |
| **Precision** | FP32 (standard) |
| **Image Size** | 640×640 |
| **Optimizer** | AdamW |
| **Early Stopping** | 20 epochs patience |

In [ ]:
from ultralytics import YOLO
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Using device: {device}")
if device == 'cpu':
    print("⚠️ Training on CPU will be very slow. Use a GPU runtime!")

# Load YOLO26n (nano — optimized for mobile)
model_fp32 = YOLO('yolo26n.pt')
print("✅ YOLO26n model loaded!")

print("\n🚀 Starting FP32 training...")
print("   This may take 30-60 minutes on a T4 GPU.\n")

results_fp32 = model_fp32.train(
    data=DATA_YAML,           # Pre-converted YOLO dataset
    epochs=100,               # Max epochs
    imgsz=640,                # Input image size
    batch=16,                 # Batch size (adjust for GPU memory)
    name='fall_detection_fp32',
    patience=20,              # Early stopping patience
    verbose=True,
    device=device,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,              # Cosine LR schedule
    augment=True,             # Data augmentation
    hsv_h=0.015,              # HSV-Hue augmentation
    hsv_s=0.7,                # HSV-Saturation augmentation
    hsv_v=0.4,                # HSV-Value augmentation
    degrees=10.0,             # Rotation augmentation
    translate=0.1,            # Translation augmentation
    scale=0.5,                # Scale augmentation
    fliplr=0.5,               # Horizontal flip
    mosaic=1.0,               # Mosaic augmentation
)

print(f"\n✅ FP32 Training completed!")
print(f"📁 Results saved in: {results_fp32.save_dir}")

In [ ]:
from pathlib import Path

# Load the best checkpoint
best_fp32_path = Path(results_fp32.save_dir) / 'weights' / 'best.pt'
print(f"📥 Loading best FP32 model: {best_fp32_path}")
best_fp32 = YOLO(best_fp32_path)

# Evaluate on validation set
print("\n🔍 Evaluating FP32 model...")
metrics_fp32 = best_fp32.val(data=DATA_YAML, verbose=True)

print("\n" + "=" * 55)
print("📊 FP32 Model — Evaluation Results")
print("=" * 55)
print(f"  mAP50       : {metrics_fp32.box.map50:.4f}  (precision @ IoU=0.5)")
print(f"  mAP50-95    : {metrics_fp32.box.map:.4f}   (global precision)")
print(f"  Precision   : {metrics_fp32.box.mp:.4f}")
print(f"  Recall      : {metrics_fp32.box.mr:.4f}")
print("=" * 55)

map50 = metrics_fp32.box.map50
if map50 >= 0.7:
    print("🟢 Excellent precision! Model is ready for deployment.")
elif map50 >= 0.5:
    print("🟡 Good precision. Acceptable for mobile deployment.")
elif map50 >= 0.3:
    print("🟠 Average precision. Consider more training data or epochs.")
else:
    print("🔴 Low precision. Significant improvements needed.")

---
# Model Training — INT8 (Quantization-Aware Training)

| Property | Value |
|----------|-------|
| **Model** | YOLO26n (Nano) |
| **Precision** | INT8 (QAT) |
| **Benefit** | ~2-4× smaller, ~2× faster inference |
| **Trade-off** | Slight accuracy reduction |

In [ ]:
# ── Model 2: Quantization-Aware Training (QAT) ────────────────

print("🔬 Starting Quantization-Aware Training...")
print("   Fake-quantization nodes simulate INT8 precision during training.")
print("   The model learns weights that tolerate reduced precision.\n")

model_qat = YOLO('yolo26n.pt')
QAT_SUCCESS = False

try:
    results_qat = model_qat.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,
        batch=16,
        name='fall_detection_qat',
        patience=20,
        verbose=True,
        device=device,
        optimizer='AdamW',
        lr0=0.0005,            # Lower LR for QAT stability
        cos_lr=True,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
        int8=True,             # ← Enable QAT
    )
    QAT_SUCCESS = True
    print(f"\n✅ QAT Training completed!")
    print(f"📁 Results saved in: {results_qat.save_dir}")

except Exception as e:
    print(f"\n⚠️ QAT training failed: {e}")
    print("📌 Falling back to standard training + PTQ export...\n")

    model_qat = YOLO('yolo26n.pt')
    results_qat = model_qat.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,
        batch=16,
        name='fall_detection_int8_fallback',
        patience=20,
        verbose=True,
        device=device,
        optimizer='AdamW',
        lr0=0.001,
        cos_lr=True,
        augment=True,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
    )
    print(f"\n✅ Fallback training completed.")
    print(f"   INT8 will be applied via PTQ during TFLite export.")
    print(f"📁 Results saved in: {results_qat.save_dir}")

In [ ]:
# Load the best QAT checkpoint
best_qat_path = Path(results_qat.save_dir) / 'weights' / 'best.pt'
print(f"📥 Loading best QAT model: {best_qat_path}")
best_qat = YOLO(best_qat_path)

# Evaluate on validation set
print("\n🔍 Evaluating QAT/INT8 model...")
metrics_qat = best_qat.val(data=DATA_YAML, verbose=True)

print("\n" + "=" * 55)
print("📊 QAT/INT8 Model — Evaluation Results")
print("=" * 55)
print(f"  mAP50       : {metrics_qat.box.map50:.4f}  (precision @ IoU=0.5)")
print(f"  mAP50-95    : {metrics_qat.box.map:.4f}   (global precision)")
print(f"  Precision   : {metrics_qat.box.mp:.4f}")
print(f"  Recall      : {metrics_qat.box.mr:.4f}")
print("=" * 55)

if QAT_SUCCESS:
    print("✅ True QAT model — accuracy should be close to FP32.")
else:
    print("ℹ️ Fallback model — PTQ will be applied during export.")

---
# Model Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Metrics Comparison Table ──────────────────────────────────
print("=" * 65)
print(f"{'Metric':<20} {'FP32':>15} {'QAT/INT8':>15} {'Δ':>10}")
print("=" * 65)

metrics_pairs = [
    ("mAP50",     metrics_fp32.box.map50, metrics_qat.box.map50),
    ("mAP50-95",  metrics_fp32.box.map,   metrics_qat.box.map),
    ("Precision",  metrics_fp32.box.mp,    metrics_qat.box.mp),
    ("Recall",     metrics_fp32.box.mr,    metrics_qat.box.mr),
]

for name, fp32_val, qat_val in metrics_pairs:
    delta = qat_val - fp32_val
    sign = "+" if delta >= 0 else ""
    print(f"  {name:<18} {fp32_val:>14.4f} {qat_val:>14.4f} {sign}{delta:>9.4f}")

print("=" * 65)

# ── Visual Comparison ─────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('📊 FP32 vs QAT/INT8 — Performance Comparison', fontsize=14, fontweight='bold')

colors = ['#4A90D9', '#E8734A']
metric_names = [m[0] for m in metrics_pairs]
fp32_vals = [m[1] for m in metrics_pairs]
qat_vals = [m[2] for m in metrics_pairs]

for i, (ax, name) in enumerate(zip(axes, metric_names)):
    bars = ax.bar(['FP32', 'INT8'], [fp32_vals[i], qat_vals[i]], color=colors, width=0.5, edgecolor='white', linewidth=1.5)
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, [fp32_vals[i], qat_vals[i]]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('📈 Training Curves — FP32 vs QAT/INT8', fontsize=16, fontweight='bold')

curve_configs = [
    ('train/box_loss',        'Box Loss',            0, 0),
    ('train/cls_loss',        'Classification Loss',  0, 1),
    ('train/dfl_loss',        'DFL Loss',             0, 2),
    ('metrics/mAP50(B)',      'mAP50',                1, 0),
    ('metrics/precision(B)',  'Precision',             1, 1),
    ('metrics/recall(B)',     'Recall',                1, 2),
]

for results, label, color in [(results_fp32, 'FP32', '#4A90D9'), (results_qat, 'QAT/INT8', '#E8734A')]:
    csv_path = Path(results.save_dir) / 'results.csv'
    if not csv_path.exists():
        print(f"⚠️ Results CSV not found for {label}")
        continue

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    for col, title, row, col_idx in curve_configs:
        ax = axes[row, col_idx]
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=2, label=label,
                    marker='o', markersize=2, alpha=0.8)
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.legend()
            ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✅ Training curves displayed!")